# Run the federated temporal experiment

This is the only control workbook needed to reproduce the study. The complete run evaluates all registered scratch, supplied-foundation, CheMeleon, continual-learning, freezing, learning-rate, replay, similarity, and single-task strategies across four rolling origins and the crossed 5 × 5 seed design.

Before starting, create and activate the environment described in `README.md`. A complete run is a substantial multi-day computation and should have at least 100 GB of free disk space at launch. Redundant checkpoints are pruned throughout, and trained models are deleted only after prediction coverage and the final results workbook have both been verified.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import shutil
import subprocess
import sys

ROOT = Path.cwd()
required = [
    ROOT / "data" / "data.csv",
    ROOT / "data" / "federated_model.pt",
    ROOT / "pipeline" / "portable_run.py",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing package files: {missing}")
if shutil.which("chemprop") is None:
    raise RuntimeError("Chemprop is not available. Activate the conda environment from README.md.")
free_gb = shutil.disk_usage(ROOT).free / 1024**3
print(f"Python: {sys.version.split()[0]}")
print(f"Chemprop: {version('chemprop')}")
print(f"Free disk space: {free_gb:.1f} GB")
if free_gb < 100:
    print("WARNING: a complete run should start with at least 100 GB free.")
else:
    print("Preflight checks passed.")

## Configuration

Leave `STRATEGIES = None` for the complete, fairness-symmetric experiment. To run a deliberately smaller custom study, replace it with exact strategy names from `pipeline/run_temporal_experiment.py`. A partial run is analyzed only against the strategies actually requested plus any required matched baseline.

Leave `KEEP_MODELS = False` for disk-safe operation. Set it to `True` only when checkpoints must be retained for later incremental method additions.

In [ ]:
STRATEGIES = None  # None = all 29 registered strategies
KEEP_MODELS = False
OVERWRITE = False  # False safely resumes an interrupted local run

print("Run type:", "complete 29-strategy study" if STRATEGIES is None else STRATEGIES)
print("Models retained after verification:", KEEP_MODELS)
print("Existing compatible work will be", "replaced" if OVERWRITE else "resumed")

## Start

Run the next cell once. Progress and Chemprop output are written to `experiment_run.log`. The command is resumable after interruption. When training finishes, it automatically computes metrics and significance, builds and executes `RESULTS_WORKBOOK.ipynb`, verifies every prediction set and notebook cell, and performs the configured disk cleanup.

In [ ]:
command = [sys.executable, str(ROOT / "pipeline" / "portable_run.py")]
if STRATEGIES:
    command.extend(["--strategies", *STRATEGIES])
if KEEP_MODELS:
    command.append("--keep-models")
if OVERWRITE:
    command.append("--overwrite")

print("Starting:", " ".join(command))
subprocess.run(command, cwd=ROOT, check=True)
print("Finished successfully. Open RESULTS_WORKBOOK.ipynb.")

In [ ]:
import json
from IPython.display import display, Markdown
status = json.loads((ROOT / "run_status.json").read_text())
display(status)
if status.get("state") == "complete":
    display(Markdown("[Open the completed results workbook](RESULTS_WORKBOOK.ipynb)"))